Setup the mlflow connection

In [1]:
from pathlib import Path
import os

# notebook path: notebooks/mlflow -> project root is ../../
project_root = Path.cwd().resolve().parents[1]
os.chdir(project_root)

print("project_root:", project_root)
print("cwd:", Path.cwd())

import notebooks.mlflow.mlflow_helpers as mh

exp = mh.setup_connection(root_path=".")
print("experiment:", exp.name)

project_root: /home/vlr/Workspaces/Topographic/ConTopo
cwd: /home/vlr/Workspaces/Topographic/ConTopo
experiment: contopo


Select a specific metalearner

In [3]:
meta = mh.get_metalearner_list(exp)
print(meta.columns)
# Actually I don't need to merge them with the ensembles, because the csv is in the metalearners too
meta = meta.with_columns(
    pl.col("params.component_run_ids_csv").str.split(",").alias("components_list")
)

meta_one = meta.filter(
    (pl.col("tags.rho") == "0.0")
    & (pl.col("tags.split") == "test")
    & (pl.col("tags.feature_type") == "embeddings+profiles")
    & (pl.col("tags.similarity_metric") == "cosine")
    & (pl.col("tags.behaviour") == "meta_mlp_2")
)
meta_one

['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'metrics.val_acc', 'metrics.holdout_acc', 'metrics.train_acc', 'metrics.holdout_loss', 'params.standardization_applied', 'params.num_components', 'params.meta_split_train', 'params.adapter_batch_size', 'params.meta_split_val', 'params.adapter_architecture', 'params.meta_split_seed', 'params.adapter_lr', 'params.component_run_ids_csv', 'params.adapter_epochs', 'tags.rho', 'tags.feature_type', 'tags.mlflow.user', 'tags.mlflow.runName', 'tags.component_set_hash', 'tags.split', 'tags.meta_split_seed', 'tags.behaviour', 'tags.num_components', 'tags.behaviour_input_hash', 'tags.similarity_metric', 'tags.mlflow.source.name', 'tags.mlflow.source.type', 'tags.mlflow.source.git.commit', 'tags.ensemble_name', 'tags.kind', 'tags.component_run_ids_csv']


run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.val_acc,metrics.holdout_acc,metrics.train_acc,metrics.holdout_loss,params.standardization_applied,params.num_components,params.meta_split_train,params.adapter_batch_size,params.meta_split_val,params.adapter_architecture,params.meta_split_seed,params.adapter_lr,params.component_run_ids_csv,params.adapter_epochs,tags.rho,tags.feature_type,tags.mlflow.user,tags.mlflow.runName,tags.component_set_hash,tags.split,tags.meta_split_seed,tags.behaviour,tags.num_components,tags.behaviour_input_hash,tags.similarity_metric,tags.mlflow.source.name,tags.mlflow.source.type,tags.mlflow.source.git.commit,tags.ensemble_name,tags.kind,tags.component_run_ids_csv,components_list
str,str,str,str,"datetime[ns, UTC]","datetime[ns, UTC]",f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str]
"""d55afd66b2444693bb29c13f621028…","""1""","""FINISHED""","""file:///home/vlr/Workspaces/To…",2026-03-21 13:23:14.493 UTC,2026-03-21 13:23:24.185 UTC,0.5815,0.56,0.634167,1.20448,"""True""","""2""","""0.6""","""256""","""0.2""","""Linear""","""42""","""0.001""","""188d9d0a137246b9906f478022a6a2…","""50""","""0.0""","""embeddings+profiles""","""vlr""","""ens_loss_type_cross_entropy_to…","""0a5395d516e5174b""","""test""","""42""","""meta_mlp_2""","""2""","""ad44c90377759311""","""cosine""","""scripts/05_train_adapters.py""","""LOCAL""","""8033a301b16c76aaa584aff248be1f…","""ens_loss_type_cross_entropy_to…","""metalearner""","""188d9d0a137246b9906f478022a6a2…","[""188d9d0a137246b9906f478022a6a293"", ""5c7d2b13b2564893a2d11d1b3784fd95""]"


Retrieve the first metalearner's input, and the corresponding raw embeddings from the component models.
It is pointless to compare now the models' output and the metalearner input, since the concatenated features are normalised including also the rdm, not calculated yet at this point

In [4]:
_, adapter_data = mh.load_adapter_inputs(meta_one["run_id"].to_list()[0])

# The original dataset index of X_train[0]
original_idx = int(adapter_data["train_idx"][0])
print(f"X_train[0] corresponds to original dataset index: {original_idx}")

components = meta_one["components_list"].to_list()[0]
components

# # Now load the raw inference embeddings for any component model and compare
embs = [
    mh.load_inference_results_from_model_run_id(exp, c)[1]["embeddings"][original_idx]
    for c in components
]
# raw_emb = tensors['embeddings'][original_idx]
# print(f"Raw embedding[:5]: {emb[:5]}")
for emb in embs:
    print(f"Raw embedding[:5]: {emb[:5]}")

X_train[0] corresponds to original dataset index: 8132
Raw embedding[:5]: [-0.37649575  0.13233483  0.6545672  -1.2982204   0.16532543]
Raw embedding[:5]: [-2.2166252   0.34082222 -0.6901809  -0.39436704 -0.34252083]


Retrieve the anchors using the same params as the ones used for metalearners

In [10]:
from src.data.anchors import get_or_create_anchors
from src.data.loaders import get_split_labels
from src.config.paths import get_cache_dir

split = "test"
labels = get_split_labels(cfg, split)

sel = cfg.pipeline.anchors
anchors = get_or_create_anchors(
    labels=labels,
    source_split=sel.source_split,
    per_class=sel.per_class,
    strategy=sel.strategy,
    order_by=sel.order_by,
    num_classes=cfg.dataset.num_classes,
    artifacts_root=str(get_cache_dir(cfg)),
    dataset_name=cfg.dataset.name,
)

# # 4. Access the specific properties you need
anchor_indices = anchors["anchor_indices"]
# anchor_labels = anchors["anchor_labels"]
# spec_hash = anchors["spec_hash"]
print(f"Loaded {len(anchor_indices)} anchors")
assert len(anchor_indices) == len(set(anchor_indices))

Loaded 1000 anchors


/home/vlr/Workspaces/Topographic/ConTopo/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Calculate the class-averaged cosine similarity between the selected image and the anchors

In [11]:
import torch.nn.functional as F
import torch

class_similarities = {}

for c in components:
    # 1. Fetch
    _, tensors = mh.load_inference_results_from_model_run_id(exp, c)
    # N num of embeddings
    # D length of embeddings
    # K num of anchors
    # C num of classes
    embeddings = torch.from_numpy(tensors["embeddings"])  # [N, D]

    # 2. Slice anchors
    current_anchors = embeddings[anchors["anchor_indices"]]  # Shape: [K, D]

    # 3. Target vector
    first = embeddings[original_idx]  # Shape: [D]

    # 3b. Get cosine similarity
    similarities = F.cosine_similarity(
        current_anchors, first.unsqueeze(0), dim=1
    )  # Shape: [K]

    # 4. Group anchors by class
    per_class = anchors["spec"]["per_class"]
    num_classes = anchors["spec"]["num_classes"]
    anchors_by_class = similarities.view(num_classes, per_class)  # Shape: [C, K/C]

    # 5. For a given image (in this case the first), average over them and save the result
    class_similarities[c] = anchors_by_class.mean(dim=1)

class_similarities

{'188d9d0a137246b9906f478022a6a293': tensor([ 0.7420,  0.7476, -0.0201, -0.2099, -0.2661, -0.1751, -0.3528,  0.0756,
          0.8762,  0.6185]),
 '5c7d2b13b2564893a2d11d1b3784fd95': tensor([ 0.6333,  0.3582,  0.0676, -0.0277, -0.1323, -0.0653, -0.0434, -0.0557,
          0.7992,  0.4207])}

Calculate RDMs across

In [7]:
from scipy.stats import pearsonr
import numpy as np

# Mask out the true class from each model's profile
true_label = int(adapter_data["y_train"][0])
masked_similarities = {}
for c, profile in class_similarities.items():
    masked_similarities[c] = torch.cat(
        [profile[:true_label], profile[true_label + 1 :]]
    )  # [C-1]

# Build RDM
component_ids = list(masked_similarities.keys())
M = len(component_ids)
rdm = np.zeros((M, M))

for i in range(M):
    for j in range(M):
        if i == j:
            rdm[i, j] = 0.0
        else:
            profile_i = masked_similarities[component_ids[i]].numpy()
            profile_j = masked_similarities[component_ids[j]].numpy()
            corr, _ = pearsonr(profile_i, profile_j)
            rdm[i, j] = 1.0 - corr

print(f"RDM Shape: {rdm.shape}")
print(rdm)

# Print first 5 elements of each model's masked profile + the full profile value
for c in component_ids:
    prof = masked_similarities[c]
    # Mean-center and normalise (like Pearson does internally)
    centered = prof - prof.mean()
    normed = centered / centered.norm().clamp_min(1e-8)
    print(f"\nModel {c[:8]}...")
    print(f"  Masked profile[:5]:     {prof[:5].numpy()}")
    print(f"  Normalised profile[:5]: {normed[:5].numpy()}")

# Extract upper triangle
upper_triangle_indices = np.triu_indices_from(rdm, k=1)
simcat = rdm[upper_triangle_indices]
print(f"\nExtracted {len(simcat)} unique pairwise dissimilarities:")
print(simcat)

RDM Shape: (2, 2)
[[0.         0.06766683]
 [0.06766683 0.        ]]

Model 188d9d0a...
  Masked profile[:5]:     [ 0.7419572   0.74764043 -0.0201296  -0.20985344 -0.26608655]
  Normalised profile[:5]: [ 0.48189276  0.4863598  -0.11711066 -0.2662344  -0.31043383]

Model 5c7d2b13...
  Masked profile[:5]:     [ 0.6332972   0.3582181   0.06763323 -0.02773961 -0.13231038]
  Normalised profile[:5]: [ 0.6572551   0.29918462 -0.07906974 -0.20321658 -0.33933637]

Extracted 1 unique pairwise dissimilarities:
[0.06766683]


In [8]:
# Convert embeddings to numpy arrays and concatenate
raw_base = np.concatenate([e for e in embs])  # Shape: [512]

# Glue the RDM profile feature at the end
full_raw_input = np.concatenate([raw_base, simcat])  # Shape: [513]

print(f"Reconstructed full raw input (shape {full_raw_input.shape})")

Reconstructed full raw input (shape (513,))


In [9]:
# 1. Grab the training standardisation stats
mean = adapter_data["standardize_mean"][0]  # Shape: [513]
std = adapter_data["standardize_std"][0]  # Shape: [513]

# 2. Standardise our manually rebuilt input
manual_standardised = (full_raw_input - mean) / (std + 1e-6)

# 3. Fetch what script 05 genuinely stored for X_train[0]
actual_xtrain_0 = adapter_data["X_train"][0]

# 4. Prove they match (np.allclose safely ignores tiny float32 precision rounding)
is_match = np.allclose(manual_standardised, actual_xtrain_0, atol=1e-4)

print(f"Manual input perfectly matches MLflow X_train[0]: {is_match}")

if not is_match:
    diff = np.abs(manual_standardised - actual_xtrain_0)
    print(f"\nMax difference found: {diff.max():.8f}")

Manual input perfectly matches MLflow X_train[0]: True
